In [1]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

In [3]:
llm = ChatGroq(
    api_key="", # enter your api key from groq
    model="llama-3.1-8b-instant",
    temperature=0
)

In [4]:
class ChatState(TypedDict):
    messages: List[str]     # the running conversation
    name: str               # remembered user name
    preferences: str        # remembered preferences

In [5]:
def chatbot(state: ChatState) -> dict:
    user_msg = state['messages'][-1]

    # very simple memory rules for a beginner demo
    name = state.get('name', '')
    prefs = state.get('preferences', '')

    if 'my name is' in user_msg.lower():
        name = user_msg.lower().split('my name is')[-1].strip().title()
    if 'i like' in user_msg.lower():
        prefs = user_msg.lower().split('i like')[-1].strip()

    context = f"The user's name is {name}. They like {prefs}."
    reply = llm.invoke(context + ' Reply to: ' + user_msg).content

    return {'messages': state['messages'] + [reply],
            'name': name, 'preferences': prefs}

In [6]:
builder = StateGraph(ChatState)
builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)
graph = builder.compile()

In [7]:
state = {'messages': [], 'name': '', 'preferences': ''}

while True:
    user = input('You: ')
    if user.lower() == 'quit':
        break
    state['messages'].append(user)
    state = graph.invoke(state)
    print('Bot:', state['messages'][-1])

You: hello, i'm medha
Bot: Hello Medha, it's nice to meet you. What brings you here today?
You: quit
